# 🤖 Taller: Machine Learning para detectar zonas urbanas — **en tu navegador**

**Curso de análisis de imágenes satelitales · Dr. Abel Coronado**

En este taller vas a **entrenar un modelo de aprendizaje automático** que distingue lo urbano de lo no urbano en una imagen satelital de **TODO el estado de Aguascalientes** — y todo el camino que eso requiere: preparar la imagen, convertir píxeles en objetos, etiquetarlos con datos oficiales de INEGI, y producir un **mapa clasificado por la máquina**. La segmentación, las estadísticas y la red neuronal son las piezas; **el aprendizaje automático es el destino**.

---

## 🧭 Antes de empezar: ¿dónde estás parado?

Esto que ves es **JupyterLab**, el entorno de trabajo estándar de la ciencia de datos. Pero esta versión tiene algo especial: **no está instalada en tu computadora ni corre en un servidor**. Todo el laboratorio — Python, las bibliotecas científicas y geoespaciales, tus datos — vive **dentro de esta pestaña del navegador**, gracias a una tecnología llamada **WebAssembly**.

¿Qué significa eso para ti?

| | |
|---|---|
| 🚫 **Nada que instalar** | No necesitas permisos de administrador ni descargar programas |
| 🔒 **Tus datos no viajan** | El procesamiento ocurre en TU equipo; nada se sube a ninguna nube |
| 🧹 **Cero rastro** | Cierras la pestaña y no queda nada instalado |
| ⚡ **Es Python de verdad** | El mismo código que usarías en un servidor profesional |

**Cómo se usa:** cada bloque gris es una *celda* de código. Haz clic en ella y presiona **Shift + Enter** para ejecutarla. Ve en orden, de arriba hacia abajo. La primera celda tarda unos segundos extra (carga las bibliotecas científicas la primera vez) — es normal.

Empecemos por comprobar que no te estoy mintiendo: 👇

In [ ]:
# ¿Dónde está corriendo este Python? Vamos a preguntárselo directamente.
import sys
import platform

print(f"Versión de Python : {sys.version.split()[0]}")
print(f"Sistema operativo : {sys.platform!r}")
print(f"Arquitectura      : {platform.machine()!r}")
print()
if sys.platform == "emscripten":
    print("✅ 'emscripten' y 'wasm32' significan: este Python corre en WebAssembly,")
    print("   DENTRO de tu navegador. No hay servidor. El laboratorio eres tú. 🚀")
else:
    print("ℹ️ Estás corriendo este cuaderno fuera del navegador (modo local).")

---
## 📚 Teoría 1: la imagen — TODO el estado de Aguascalientes

Trabajaremos con la escena completa del estado (el rectángulo de ~111 × 100 km que lo contiene, con pedacitos de Zacatecas y Jalisco en las orillas). Esta imagen tiene una historia que vale la pena conocer:

- **La fuente es Sentinel-2**, la misión del programa europeo Copernicus que fotografía toda la Tierra cada ~5 días con píxeles de 10 metros y 12 bandas espectrales. Sus datos son **públicos y gratuitos**.
- Usamos su **geomediana del año 2020**: se toman *todas* las imágenes del año y se calcula, píxel por píxel, un valor mediano robusto. Resultado: el año entero en una sola imagen, **sin nubes ni sombras**.
- Esa geomediana completa pesa **2.5 GB** — más de lo que cabe en la memoria de una pestaña de navegador. Así que la **remuestreamos a "calidad Landsat"**: píxeles de **30 metros** y las **6 bandas clásicas** — azul, verde, rojo, infrarrojo cercano (NIR) e infrarrojos de onda corta (SWIR1 y SWIR2). Es exactamente la combinación con la que la familia **Landsat** lleva 50 años observando la Tierra, y deja la imagen en **176 MB**: el estado entero, dentro de tu navegador.

💡 Las bandas infrarrojas hacen la magia: vegetación, agua, suelo desnudo y concreto — que en color natural se confunden — en NIR y SWIR se separan con claridad. Por eso clasificamos con 6 números por píxel y no con 3.

Cada píxel es entonces un **vector de 6 mediciones** de un cuadrito de 30×30 metros. Y para que el **linaje** sea completo: la receta exacta del remuestreo está publicada en el repositorio del curso (`wasm/qa/landsat_sim.py`) — puedes auditar cada paso desde el dato crudo hasta lo que ves aquí.

Traigamos la imagen (en el **kit offline** ya viene contigo en `mis_datos`; en línea se descarga una sola vez):

In [ ]:
# Paso 1a — Preparar las herramientas y traer la imagen estatal
%pip install -q shepherd-wasm
import os
import time
import numpy as np
import rasterio
from rasterio import features
import matplotlib.pyplot as plt
import scipy.ndimage
import sklearn.cluster

HF = "https://huggingface.co/datasets/abxda/portable-satelital/resolve/main/taller/"

async def carga_dato(nombre):
    """Trae un dato del taller a /tmp: primero tu carpeta mis_datos (kit
    offline), y si no está, lo descarga de internet una sola vez."""
    destino = "/tmp/" + nombre
    if os.path.exists(destino):
        return destino
    from pyodide.http import pyfetch
    for url in ("/mis_datos/" + nombre, HF + nombre):
        try:
            r = await pyfetch(url)
            if r.ok:
                with open(destino, "wb") as f:
                    f.write(await r.bytes())
                return destino
        except Exception:
            pass
    raise FileNotFoundError(f"no pude traer {nombre}")

t0 = time.time()
ruta = await carga_dato("ags_landsat_30m.tif")
with rasterio.open(ruta) as src:
    img = src.read().astype(np.float32)   # (bandas, filas, columnas)
    transform, crs, nodata = src.transform, src.crs, src.nodata
    nombres_bandas = src.descriptions

n_bandas, alto, ancho = img.shape
print(f"✓ imagen estatal lista en {time.time()-t0:.1f} s: {n_bandas} bandas × {alto} filas × {ancho} columnas")
print(f"  cada píxel cubre 30 m × 30 m → la escena mide ≈ {ancho*30/1000:.0f} × {alto*30/1000:.0f} km: todo Aguascalientes")
print(f"  bandas: {', '.join(nombres_bandas)}")
print(f"  banda roja: mín={img[2].min():.0f}  máx={img[2].max():.0f}  media={img[2].mean():.0f}")
print(f"  banda NIR : mín={img[3].min():.0f}  máx={img[3].max():.0f}  media={img[3].mean():.0f}")
print()
print("💡 Para la máquina la 'imagen' son 73 millones de números. Nada más — y nada menos.")

In [ ]:
# Paso 1b — Verla como la verían tus ojos (composición en color natural)
# Bandas rojo, verde y azul; estiramos contraste entre percentiles 2-98
rgb = np.stack([img[2], img[1], img[0]], axis=-1)
p2, p98 = np.percentile(rgb[::5, ::5], (2, 98))
rgb = np.clip((rgb - p2) / (p98 - p2), 0, 1)

plt.figure(figsize=(7.5, 7))
plt.imshow(rgb[::3, ::3])   # para dibujar rápido mostramos 1 de cada 3 píxeles
plt.title("Aguascalientes completo — 'calidad Landsat' (30 m) derivada de Sentinel-2")
plt.axis("off")
plt.show()
print("💡 Ahí está la mancha de la capital, los valles agrícolas del centro,")
print("   la sierra al poniente y la presa Calles al norte. Ya no es un recorte:")
print("   es TU estado completo, y lo vas a clasificar tú.")

---
## 📚 Teoría 2: ¿por qué "segmentar"? Del píxel al objeto

Si quisiéramos clasificar esta imagen píxel por píxel ("¿este píxel es urbano o agrícola?"), tendríamos dos problemas: **ruido** (píxeles aislados mal clasificados, efecto "sal y pimienta") y **falta de contexto** (un píxel gris puede ser una calle, un techo o suelo desnudo — solo, no se sabe).

La alternativa es el enfoque **orientado a objetos (GEOBIA)**: primero agrupamos los píxeles en **segmentos** — regiones contiguas espectralmente homogéneas que corresponden a *cosas* del territorio: una parcela, una manzana, un cuerpo de agua. Después clasificamos los segmentos, no los píxeles.

Usaremos el algoritmo de **Shepherd, Bunting y Dymond (2019)** (*Remote Sensing* 11(6):658), el mismo que usan agencias de monitoreo territorial. Funciona en 3 pasos:

1. **Siembra (K-means):** agrupa los píxeles en ~60 "familias espectrales" según sus 6 bandas — sin importar dónde están.
2. **Aglomerado (clumping):** los píxeles *vecinos* que cayeron en la misma familia se unen en grupos contiguos. Salen miles de grupitos.
3. **Eliminación iterativa:** los grupos demasiado chicos (menos de `minSegmentSize` píxeles) se fusionan, del más pequeño al más grande, con el vecino espectralmente más parecido. Quedan solo objetos de tamaño razonable.

La implementación que vas a ejecutar (`shepherd_wasm`) está **validada bit a bit** contra la implementación de referencia (`pyshepseg`): produce exactamente los mismos segmentos.

⏱️ *Una nota de expectativas: vamos a segmentar 12 millones de píxeles dentro de una pestaña. El paso completo toma ~3 minutos — aprovecha para estirarte; tu navegador está haciendo trabajo de servidor.*

Veamos el **paso 1** con nuestros propios ojos:

In [ ]:
# PASO 1 — Siembra: K-means agrupa los píxeles en 60 familias espectrales
import shepherd_wasm

t0 = time.time()
km = shepherd_wasm.fitSpectralClusters(img, numClusters=60, subsamplePcnt=1,
                                       imgNullVal=nodata, fixedKMeansInit=True)
clusters = shepherd_wasm.applySpectralClusters(km, img, nodata)
print(f"K-means listo en {time.time()-t0:.1f} s — cada píxel tiene ahora una 'familia' (1 a 60)")

plt.figure(figsize=(12, 5.5))
plt.subplot(1, 2, 1); plt.imshow(rgb[::3, ::3]); plt.title("El estado"); plt.axis("off")
plt.subplot(1, 2, 2); plt.imshow(clusters[::3, ::3], cmap="tab20", interpolation="nearest")
plt.title("Paso 1: familias espectrales (colores = familias)"); plt.axis("off")
plt.tight_layout(); plt.show()
print("💡 Observa: las familias capturan tipos de cobertura, pero quedan 'salpicadas'.")
print("   Los pasos 2 y 3 convierten esta sal y pimienta en objetos limpios.")

In [ ]:
# PASOS 2 y 3 — Aglomerar y depurar: la segmentación completa
# (reutilizamos el K-means que ya ajustamos: kmeansObj=km)
# ⏱️ paciencia: ~3 minutos — 12 millones de píxeles DENTRO de tu navegador
t0 = time.time()
res = shepherd_wasm.doShepherdSegmentation(
    img,
    kmeansObj=km,          # paso 1 ya hecho
    minSegmentSize=50,     # tamaño mínimo de objeto: 50 px = 4.5 ha a 30 m
    imgNullVal=nodata)
seg = res.segimg

print(f"Segmentación completa en {time.time()-t0:.1f} s — ¡dentro de tu navegador!")
print(f"  · objetos finales              : {int(seg.max()):,}")
print(f"  · píxeles sueltos absorbidos   : {res.singlePixelsEliminated:,}")
print(f"  · grupitos chicos fusionados   : {res.smallSegmentsEliminated:,}")
print(f"  · umbral espectral de fusión   : {res.maxSpectralDiff:.0f} (calculado automáticamente)")

In [ ]:
# Visualicemos el resultado. A escala estatal las fronteras no se aprecian,
# así que nos acercamos a la capital (el recuadro amarillo).
from scipy import ndimage
import matplotlib.patches as mpatches

cy, cx = 2266, 2029          # centro de la mancha urbana de la capital
V = 350                      # media ventana: 350 px = 10.5 km
r0, r1, c0, c1 = cy - V, cy + V, cx - V, cx + V

bordes = (ndimage.maximum_filter(seg[r0:r1, c0:c1], size=2)
          != ndimage.minimum_filter(seg[r0:r1, c0:c1], size=2))
vis = rgb[r0:r1, c0:c1].copy()
vis[bordes] = [1, 1, 0]      # fronteras en amarillo

fig, ax = plt.subplots(1, 2, figsize=(12.5, 6))
ax[0].imshow(rgb[::3, ::3])
ax[0].add_patch(mpatches.Rectangle((c0/3, r0/3), (c1-c0)/3, (r1-r0)/3,
                                   fill=False, edgecolor="yellow", linewidth=2))
ax[0].set_title("El estado (recuadro = acercamiento)"); ax[0].axis("off")
ax[1].imshow(vis)
ax[1].set_title(f"{int(seg.max()):,} objetos en el estado — detalle: la capital")
ax[1].axis("off")
plt.tight_layout(); plt.show()
print("💡 Ya no son píxeles: son parcelas, manzanas, presas. Objetos con sentido.")

---
## 📚 Teoría 3: cada objeto, una fila en una tabla

Para clasificar los objetos necesitamos describirlos con números: sus **estadísticas zonales** — para cada segmento, la media y desviación estándar de cada una de las 6 bandas. Eso convierte la imagen en una **tabla**: una fila por objeto, una columna por característica. Y una tabla ya es territorio conocido: es lo que come cualquier algoritmo de aprendizaje automático.

Como nuestros segmentos están perfectamente alineados al píxel, el cálculo es exacto y rapidísimo con `numpy`:

In [ ]:
import pandas as pd

t0 = time.time()
nseg = int(seg.max()) + 1
flat = seg.ravel()
n_px = np.bincount(flat, minlength=nseg)

tabla = {"segment_id": np.arange(1, nseg), "n_px": n_px[1:]}
for b in range(n_bandas):
    v = img[b].ravel()
    suma  = np.bincount(flat, weights=v,     minlength=nseg)
    suma2 = np.bincount(flat, weights=v * v, minlength=nseg)
    media = np.where(n_px > 0, suma / n_px, 0)
    var   = np.maximum(np.where(n_px > 0, suma2 / n_px, 0) - media**2, 0)
    tabla[f"b{b+1}Mean"]   = media[1:]
    tabla[f"b{b+1}StdDev"] = np.sqrt(var)[1:]

df = pd.DataFrame(tabla)
print(f"Tabla de características en {time.time()-t0:.2f} s: "
      f"{df.shape[0]:,} objetos × {df.shape[1]-2} características espectrales")
df.head()

In [ ]:
# Y de regreso al mapa: pintamos cada objeto por su reflejo en el
# infrarrojo cercano (NIR ≈ vigor de la vegetación). Con cien mil objetos el
# truco eficiente es una tabla de búsqueda: valores[seg] pinta todo de golpe.
lut_nir = np.zeros(nseg, dtype=np.float32)
lut_nir[df.segment_id.values] = df.b4Mean.values     # banda 4 = NIR
mapa_nir = lut_nir[seg]

plt.figure(figsize=(7.8, 7))
plt.imshow(mapa_nir[::3, ::3], cmap="RdYlGn")
plt.colorbar(shrink=0.7, label="NIR medio del objeto")
plt.title("Cada objeto pintado por su infrarrojo cercano\n(verde = vegetación vigorosa)")
plt.axis("off"); plt.show()
print(f"{df.shape[0]:,} objetos descritos por sus estadísticas espectrales.")
print("Cada uno tiene coordenadas reales: al final exportaremos los resultados")
print("en formatos que QGIS abre directamente.")

---
## 📚 Teoría 4: el aprendizaje automático — enseñarle al modelo con ejemplos

Ya tenemos objetos descritos por números. Falta lo importante: que la máquina aprenda a **distinguir lo urbano de lo no urbano**. Para eso necesitamos **ejemplos etiquetados** — la "verdad-terreno".

Y aquí este taller no te regala nada pre-cocinado: **la verdad-terreno la vas a construir tú**, con el linaje completo a la vista:

1. **Fuente oficial:** los polígonos de localidades del **Marco Geoestadístico de INEGI** que caen en el encuadre — te los damos tal cual, en un GeoJSON estándar con nombre y ámbito de cada localidad.
2. **Filtrar:** quedarnos solo con las localidades **urbanas**.
3. **Proyectar:** llevarlas de coordenadas geográficas (grados) al sistema de coordenadas de *nuestra* imagen (metros).
4. **Rasterizar:** pintarlas sobre la malla exacta de 30 m — cada píxel del estado queda etiquetado.
5. **Depurar y separar:** entrenamos solo con objetos *puros* (≥ 90 % de una clase), **balanceados** (mismo número de ejemplos por clase) y partidos 70/30 — el 30 % el modelo **nunca lo ve**, para calificarlo honestamente.
6. **Entrenar y evaluar:** el pipeline del curso — una red neuronal (MLP) apilada con un bosque de árboles extra-aleatorios.

Si mañana alguien te pregunta *"¿de dónde salieron las etiquetas de tu mapa?"*, la respuesta completa está en la siguiente celda: ejecutable, auditable, reproducible. **Eso es linaje.** Y todo — también el entrenamiento — ocurre dentro de tu navegador.

In [ ]:
# La verdad-terreno NO viene pre-hecha: la construimos AQUÍ, paso a paso.
import json
import geopandas as gpd

# Fuente: localidades del Marco Geoestadístico (INEGI) en el encuadre,
# en GeoJSON estándar (coordenadas geográficas WGS84)
ruta_loc = await carga_dato("localidades_encuadre.geojson")
loc = gpd.GeoDataFrame.from_features(
    json.load(open(ruta_loc, encoding="utf-8"))["features"], crs="EPSG:4326")
print(f"{len(loc)} localidades registradas en el encuadre. Una muestra:")
display(loc[["CVEGEO", "NOMGEO", "AMBITO"]].sample(6, random_state=1))

# Linaje paso 1: filtrar las URBANAS
urbanas = loc[loc.AMBITO == "Urbana"]
print(f"→ {len(urbanas)} localidades urbanas (las rurales no nos sirven de ejemplo 'urbano')")

# Linaje paso 2: proyectarlas al sistema de coordenadas de NUESTRA imagen
urbanas = urbanas.to_crs(crs)

# Linaje paso 3: rasterizarlas a la malla exacta de 30 m de la imagen
etiquetas = features.rasterize(
    ((geom, 1) for geom in urbanas.geometry),
    out_shape=seg.shape, transform=transform, fill=2, dtype="uint8")
# 1 = localidad urbana, 2 = resto del territorio

plt.figure(figsize=(7.8, 7))
plt.imshow(rgb[::3, ::3])
plt.imshow(np.where(etiquetas[::3, ::3] == 1, 1.0, np.nan),
           cmap="autumn", alpha=0.9, interpolation="nearest")
plt.title(f"Verdad-terreno construida por TI: {len(urbanas)} localidades urbanas (naranja)")
plt.axis("off"); plt.show()

# Linaje paso 4: conectar etiquetas con objetos — proporción urbana de cada uno
urb_px = np.bincount(flat, weights=(etiquetas == 1).ravel(), minlength=nseg)
df["prop_urbano"] = np.where(n_px > 0, urb_px / n_px, 0)[1:]

print(f"objetos mayormente urbanos   : {(df.prop_urbano >= 0.5).sum():,}")
print(f"objetos mayormente no urbanos: {(df.prop_urbano < 0.5).sum():,}")

In [ ]:
# Entrenamiento: el MISMO pipeline del ejercicio completo del curso
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

class StackingEstimator(BaseEstimator, TransformerMixin):
    """Apila las predicciones de un modelo como features extra para el siguiente."""
    def __init__(self, estimator):
        self.estimator = estimator
    def fit(self, X, y=None, **kw):
        self.estimator_ = clone(self.estimator); self.estimator_.fit(X, y, **kw); return self
    def transform(self, X):
        X = np.asarray(X); out = [X]
        if hasattr(self.estimator_, "predict_proba"):
            out.append(self.estimator_.predict_proba(X))
        out.append(self.estimator_.predict(X).reshape(-1, 1))
        return np.hstack(out)

# 1) Solo objetos PUROS para entrenar (>=90% de una clase): etiquetas confiables
puros = df[(df.prop_urbano >= 0.9) | (df.prop_urbano <= 0.1)].copy()
puros["clase"] = np.where(puros.prop_urbano >= 0.9, 1, 2)

# 2) Balancear: mismo número de ejemplos de cada clase (muestreo reproducible)
n_min = int(puros.clase.value_counts().min())
balanceado = pd.concat([
    puros[puros.clase == 1].sample(n=n_min, random_state=42),
    puros[puros.clase == 2].sample(n=n_min, random_state=42),
])

feat_cols = [c for c in df.columns if c.startswith("b")]   # 24 features espectrales
X, y = balanceado[feat_cols], balanceado["clase"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

t0 = time.time()
pipeline = make_pipeline(
    StandardScaler(),
    StackingEstimator(MLPClassifier(alpha=0.01, learning_rate_init=0.001,
                                    max_iter=300, random_state=42)),
    ExtraTreesClassifier(bootstrap=False, criterion="entropy", max_features=0.8,
                         n_estimators=100, random_state=42))
pipeline.fit(Xtr, ytr)
acc = accuracy_score(yte, pipeline.predict(Xte))

print(f"Entrenado en {time.time()-t0:.1f} s con {len(Xtr)} objetos ({n_min} por clase)")
print(f"Exactitud en prueba: {acc:.1%}  (sobre {len(Xte)} objetos que el modelo NUNCA vio)")
ConfusionMatrixDisplay(confusion_matrix(yte, pipeline.predict(Xte)),
                       display_labels=["urbano", "no urbano"]).plot(cmap="Blues")
plt.title("Matriz de confusión (conjunto de prueba)"); plt.tight_layout(); plt.show()

In [ ]:
# Y el gran final: clasificar TODOS los objetos y pintar el mapa ESTATAL
from matplotlib.colors import ListedColormap

t0 = time.time()
df["pred"] = pipeline.predict(df[feat_cols])
print(f"✓ el modelo clasificó los {len(df):,} objetos del estado en {time.time()-t0:.1f} s")

lut_pred = np.zeros(nseg, dtype=np.uint8)
lut_pred[df.segment_id.values] = df.pred.values
mapa_pred = lut_pred[seg]            # ráster clasificado: 1=urbano, 2=no urbano

fig, ax = plt.subplots(figsize=(8.2, 7.5))
ax.imshow(rgb[::3, ::3])
ax.imshow(np.where(mapa_pred[::3, ::3] == 1, 1.0, np.nan),
          cmap=ListedColormap(["#d62728"]), alpha=0.9, interpolation="nearest")
ax.set_title("Lo urbano de Aguascalientes según TU modelo (rojo)\n— entrenado y predicho EN TU NAVEGADOR")
ax.axis("off")
plt.savefig("mapa_clasificado.png", dpi=130, bbox_inches="tight")
plt.show()

n_u = int((df.pred == 1).sum())
km2 = float((mapa_pred == 1).sum()) * 900 / 1e6
print(f"objetos urbanos: {n_u:,}  |  superficie urbana estimada: {km2:.0f} km²")
print("💡 Compárala con la verdad-terreno: el modelo también encuentra construcción")
print("   que el registro de localidades no marca (parques industriales, fraccionamientos")
print("   nuevos, carreteras). Justo para eso entrenamos máquinas.")

---
## 📦 Llévate tus productos

Lo que acabas de producir no es una ilustración: son **productos geoespaciales reales del estado completo**, georreferenciados, listos para QGIS o para compartirse con tu equipo. La siguiente celda los guarda y te genera **botones de descarga**:

1. `mapa_urbano.tif` — el ráster clasificado del estado (GeoTIFF: 1=urbano, 2=no urbano)
2. `features_urbano.csv` — la tabla completa: objetos, características y predicciones
3. `urbano_aguascalientes.geojson` — los polígonos de lo clasificado urbano (QGIS lo abre directo)
4. `mapa_clasificado.png` — la imagen del mapa para tu presentación

---
## 🎒 Llévatelo a tu oficina: tu ambiente portable de análisis

Este kit no es solo un taller: es **tu laboratorio personal**. El procedimiento completo para hacerlo tuyo:

| Qué | Cómo |
|---|---|
| **Tus datos de entrada** | Copia tus GeoTIFF a `mis_datos\` con el Explorador — el cuaderno los ve al instante |
| **Tus resultados** | Se depositan solos en `mis_datos\salidas\` (y siempre tienes los botones ⬇) |
| **Tu cuaderno editado** | El navegador guarda una copia interna (`Ctrl+S`), pero el archivo que TÚ controlas se obtiene con **File → Download** → muévelo a `mis_datos\` |
| **Empezar de cero** | Abre **`Mi_Lienzo.ipynb`** (en el panel izquierdo): un cuaderno en blanco con el arranque ya resuelto, para TU siguiente análisis |
| **Compartir con un colega** | Copia la carpeta completa del kit a su máquina o a una USB — funciona igual, sin internet |
| **Cuando los datos crezcan** | El laboratorio portable (SatLab) corre estos mismos cuadernos con la geomediana Sentinel-2 completa (10 m, 12 bandas, 2.5 GB) — el siguiente nivel de detalle |

**La idea de fondo:** los datos viven en *tu* carpeta, los resultados regresan a *tu* carpeta, y el motor (este kit) es un acompañante que puedes borrar y volver a copiar cuando quieras. Tú tienes el control.

In [ ]:
# 📦 Tus productos, listos para llevar
# Guardamos los 4 productos del taller y generamos botones de descarga.
# Todo se creó en TU equipo; al descargar solo lo mueves a tu carpeta.
import base64
from IPython.display import HTML, display
import geopandas as gpd

# 1) ráster clasificado del estado (GeoTIFF georreferenciado, uint8 ligero)
with rasterio.open("mapa_urbano.tif", "w", driver="GTiff",
                   height=mapa_pred.shape[0], width=mapa_pred.shape[1],
                   count=1, dtype="uint8", crs=crs, transform=transform,
                   nodata=0) as dst:
    dst.write(mapa_pred, 1)

# 2) tabla de características + predicción (CSV, redondeado para que viaje ligero)
df.round(3).to_csv("features_urbano.csv", index=False)

# 3) lo URBANO como polígonos (poligonizamos solo la clase de interés,
#    no los cien mil objetos: así el archivo es ligero y útil)
mask_urb = (mapa_pred == 1).astype(np.uint8)
geoms = [{"properties": {"clase": "urbano"}, "geometry": g}
         for g, v in features.shapes(mask_urb, mask=mask_urb.astype(bool),
                                     transform=transform)]
gdf_urb = gpd.GeoDataFrame.from_features(geoms, crs=crs)
with open("urbano_aguascalientes.geojson", "w") as f:
    f.write(gdf_urb.to_json())

productos = [
    ("mapa_urbano.tif", "Ráster clasificado del estado (GeoTIFF)"),
    ("features_urbano.csv", "Tabla de características (CSV)"),
    ("urbano_aguascalientes.geojson", "Polígonos urbanos (GeoJSON para QGIS)"),
    ("mapa_clasificado.png", "Mapa (imagen PNG)"),
]

def boton_descarga(path, etiqueta):
    datos = base64.b64encode(open(path, "rb").read()).decode()
    kb = os.path.getsize(path) / 1024
    return (f'<a download="{path}" href="data:application/octet-stream;base64,{datos}" '
            f'style="display:inline-block;margin:5px 10px 5px 0;padding:10px 16px;'
            f'background:#0ea5e9;color:#fff;border-radius:9px;font-weight:600;'
            f'text-decoration:none;font-family:Segoe UI">⬇ {etiqueta} '
            f'<span style="opacity:.75;font-size:12px">({kb:,.0f} KB)</span></a>')

for p, _ in productos:
    print(f"  ✓ guardado: {p}  ({os.path.getsize(p)/1024:,.0f} KB)")
print(f"  ({len(gdf_urb):,} polígonos urbanos en el GeoJSON)")
display(HTML("<div>" + "".join(boton_descarga(p, e) for p, e in productos) + "</div>"))
print()
print("💡 El GeoTIFF y el GeoJSON conservan coordenadas reales: ábrelos en QGIS")
print("   sobre cualquier mapa base y ahí estará tu clasificación, en su lugar.")

In [ ]:
# 💾 Y además, directo a TU carpeta (solo en el kit offline)
# En el kit, este cuaderno puede DEPOSITAR los productos en mis_datos\salidas\
# — tu disco, tu carpeta, tu control. En la versión en línea usa los botones ⬇.
async def guardar_en_mi_carpeta(nombre):
    try:
        from pyodide.http import pyfetch
        with open(nombre, "rb") as f:
            r = await pyfetch("/api/guardar/" + nombre, method="POST", body=f.read())
        return r.ok
    except Exception:
        return False

guardados = []
for p, _ in productos:
    if await guardar_en_mi_carpeta(p):
        guardados.append(p)

if guardados:
    print("💾 Copiados a tu carpeta  mis_datos\\salidas\\ :")
    for p in guardados:
        print("   ✓", p)
    print("\nÁbrela con el Explorador: esos archivos ya viven en TU disco.")
else:
    print("ℹ El depósito directo funciona en el kit offline; aquí usa los botones ⬇ de arriba.")

---
## 🗂️ El taller es tuyo: usa TUS propias imágenes

Hasta aquí clasificaste el estado con nuestros datos. Pero este taller es **habilitador**: el mismo pipeline funciona con **tu** GeoTIFF multibanda y, si las tienes, **tus** etiquetas.

**Cómo darle tus datos** (dos caminos):

- **Kit offline:** copia tus archivos `.tif` a la carpeta `mis_datos\` (junto al programa del taller) con el Explorador — la celda de abajo los encontrará al instante.
- **Versión en línea:** arrastra tus `.tif` al explorador de archivos del Lab (panel izquierdo, botón ⬆️).

Requisitos: imagen **GeoTIFF multibanda** (sin comprimir o LZW). Como referencia de tamaño: todo Aguascalientes a 30 m son **12.3 millones de píxeles y corre en ~3 minutos** — esa es la zona cómoda del navegador. Para escenas mucho mayores (o la Sentinel completa de 10 m), usa el laboratorio portable. Etiquetas opcionales: GeoTIFF de 1 banda **alineado a la imagen**, con `1` (clase de interés) y `2` (resto).

In [ ]:
# 🗂️ ¿Qué datos tuyos hay disponibles?
import os
mis_archivos = []
origen = ""
try:
    from pyodide.http import pyfetch
    r = await pyfetch("/api/mis_datos")
    if r.ok:
        mis_archivos = [(d["nombre"], d["bytes"]) for d in (await r.json())]
        origen = "carpeta mis_datos (kit offline)"
except Exception:
    pass
if not mis_archivos:
    nuestros = {"mapa_urbano.tif", "features_urbano.csv",
                "urbano_aguascalientes.geojson", "mapa_clasificado.png",
                "mis_features.csv"}
    propios = [f for f in os.listdir(".")
               if f.lower().endswith((".tif", ".tiff")) and f not in nuestros]
    mis_archivos = [(f, os.path.getsize(f)) for f in propios]
    origen = "archivos subidos al explorador del Lab"

if mis_archivos:
    print(f"Encontré {len(mis_archivos)} archivo(s) tuyos en: {origen}\n")
    for n, b in mis_archivos:
        print(f"   📄 {n}   ({b/1e6:,.1f} MB)")
    print("\n👉 Copia el nombre de tu imagen (y etiquetas, si tienes) en la celda de abajo.")
else:
    print("Aún no veo archivos tuyos.")
    print("  · Kit offline: copia tus .tif a la carpeta mis_datos\\ y re-ejecuta esta celda.")
    print("  · En línea: súbelos con el botón ⬆️ del panel izquierdo y re-ejecuta.")

In [ ]:
# 🗂️ TU pipeline: escribe los nombres de TUS archivos y ejecuta
MI_IMAGEN = ""       # ej. "ejemplo_imagen.tif"   (GeoTIFF multibanda)
MIS_ETIQUETAS = ""   # ej. "ejemplo_etiquetas.tif" (opcional: 1=interés, 2=resto)

async def _trae(nombre):
    """Localiza tu archivo: subido al Lab, ya en /tmp, o en mis_datos (kit).
    Los archivos grandes van a /tmp: el disco rápido del kernel."""
    if os.path.exists(nombre):
        return nombre
    destino = "/tmp/" + nombre
    if not os.path.exists(destino):
        from pyodide.http import pyfetch
        r = await pyfetch("/mis_datos/" + nombre)
        if not r.ok:
            raise FileNotFoundError(f"no encuentro {nombre} (¿está en mis_datos\\ o subido al Lab?)")
        with open(destino, "wb") as f:
            f.write((await r.bytes()))
    return destino

if not MI_IMAGEN:
    print("👉 Escribe arriba el nombre de tu imagen (lo viste en la celda anterior) y re-ejecuta.")
else:
    ruta_mia = await _trae(MI_IMAGEN)
    with rasterio.open(ruta_mia) as s:
        tu_img = s.read().astype(np.float32)
        tu_tr, tu_crs, tu_nd = s.transform, s.crs, s.nodata
    nb_, al_, an_ = tu_img.shape
    print(f"✓ {MI_IMAGEN}: {nb_} bandas × {al_} × {an_}  (CRS: {tu_crs})")
    if al_ * an_ > 13_000_000:
        print("⚠ imagen grande para el navegador: puede tardar mucho o agotar memoria.")
        print("  Referencia: Aguascalientes completo (12.3 Mpx) corre en ~3 minutos.")
        print("  Para más que eso: recórtala, o usa el laboratorio portable.")

    t0 = time.time()
    tu_res = shepherd_wasm.doShepherdSegmentation(
        tu_img, numClusters=60, minSegmentSize=50,
        imgNullVal=tu_nd, fixedKMeansInit=True)
    tu_seg = tu_res.segimg
    print(f"✓ segmentada en {time.time()-t0:.1f} s → {int(tu_seg.max()):,} objetos")

    # visualización (RGB si hay ≥3 bandas; si no, la primera banda)
    if nb_ >= 3:
        idx = (3, 2, 1) if nb_ >= 4 else (2, 1, 0)
        tu_rgb = np.stack([tu_img[i] for i in idx], axis=-1)
    else:
        tu_rgb = np.stack([tu_img[0]] * 3, axis=-1)
    q2, q98 = np.percentile(tu_rgb, (2, 98))
    tu_rgb = np.clip((tu_rgb - q2) / max(q98 - q2, 1e-6), 0, 1)
    bb = scipy.ndimage.maximum_filter(tu_seg, 2) != scipy.ndimage.minimum_filter(tu_seg, 2)
    vis_ = tu_rgb.copy(); vis_[bb] = [1, 1, 0]
    plt.figure(figsize=(7, 7)); plt.imshow(vis_)
    plt.title(f"TUS datos, segmentados: {int(tu_seg.max()):,} objetos"); plt.axis("off"); plt.show()

    # features de TUS objetos
    nseg_ = int(tu_seg.max()) + 1
    fl_ = tu_seg.ravel()
    npx_ = np.bincount(fl_, minlength=nseg_)
    tab_ = {"segment_id": np.arange(1, nseg_), "n_px": npx_[1:]}
    for b in range(nb_):
        v = tu_img[b].ravel()
        s1 = np.bincount(fl_, weights=v, minlength=nseg_)
        s2 = np.bincount(fl_, weights=v * v, minlength=nseg_)
        m_ = np.where(npx_ > 0, s1 / npx_, 0)
        tab_[f"b{b+1}Mean"] = m_[1:]
        tab_[f"b{b+1}StdDev"] = np.sqrt(np.maximum(np.where(npx_ > 0, s2 / npx_, 0) - m_**2, 0))[1:]
    tu_df = pd.DataFrame(tab_)
    tu_df.to_csv("mis_features.csv", index=False)
    print(f"✓ tabla de TUS objetos: {tu_df.shape[0]:,} filas × {tu_df.shape[1]-2} features → mis_features.csv")

    if MIS_ETIQUETAS:
        ruta_lab_mia = await _trae(MIS_ETIQUETAS)
        with rasterio.open(ruta_lab_mia) as ls:
            tu_lab = ls.read(1)
        u_ = np.bincount(fl_, weights=(tu_lab == 1).ravel(), minlength=nseg_)
        tu_df["prop_interes"] = np.where(npx_ > 0, u_ / npx_, 0)[1:]
        puros_ = tu_df[(tu_df.prop_interes >= 0.9) | (tu_df.prop_interes <= 0.1)].copy()
        puros_["clase"] = np.where(puros_.prop_interes >= 0.9, 1, 2)
        nmin_ = int(puros_.clase.value_counts().min())
        if nmin_ < 10:
            print(f"⚠ pocos ejemplos puros por clase ({nmin_}): el modelo no será confiable.")
        bal_ = pd.concat([puros_[puros_.clase == 1].sample(nmin_, random_state=42),
                          puros_[puros_.clase == 2].sample(nmin_, random_state=42)])
        fc_ = [c for c in tu_df.columns if c.startswith("b")]
        Xtr_, Xte_, ytr_, yte_ = train_test_split(bal_[fc_], bal_["clase"],
                                                  test_size=0.3, random_state=42,
                                                  stratify=bal_["clase"])
        tu_pipe = make_pipeline(
            StandardScaler(),
            StackingEstimator(MLPClassifier(alpha=0.01, learning_rate_init=0.001,
                                            max_iter=300, random_state=42)),
            ExtraTreesClassifier(bootstrap=False, criterion="entropy",
                                 max_features=0.8, n_estimators=100, random_state=42))
        tu_pipe.fit(Xtr_, ytr_)
        acc_ = accuracy_score(yte_, tu_pipe.predict(Xte_))
        print(f"✓ TU modelo entrenado — exactitud en prueba: {acc_:.1%}")
        tu_df["pred"] = tu_pipe.predict(tu_df[fc_])
        pred_lut = np.zeros(nseg_, dtype=np.uint8)
        pred_lut[tu_df.segment_id.values] = tu_df.pred.values
        pm_ = pred_lut[tu_seg]
        plt.figure(figsize=(7, 7))
        plt.imshow(np.where(pm_ == 1, 1.0, 0.0), cmap="bwr", vmin=0, vmax=1)
        plt.title(f"TU mapa clasificado (rojo = clase de interés) — {acc_:.0%} de exactitud")
        plt.axis("off"); plt.show()
        await guardar_en_mi_carpeta("mis_features.csv")
    else:
        print("ℹ sin etiquetas: hicimos segmentación + features. Si agregas MIS_ETIQUETAS, entrenamos TU modelo.")

---
## 🧪 Experimenta tú

La celda de abajo está lista para que juegues con los **dos parámetros clave** del algoritmo — sobre el acercamiento a la capital, para que cada intento tome segundos y no minutos. Cambia los valores, ejecútala y observa cómo cambia el mapa:

- `minSegmentSize` — el tamaño mínimo de objeto en píxeles. ¿Qué pasa con 10? ¿Y con 200? *(pista: piensa qué nivel de detalle territorial necesita tu análisis)*
- `numClusters` — cuántas "familias espectrales" busca el paso 1. ¿Qué pasa con 15? ¿Y con 100?

In [ ]:
# 🧪 Tu laboratorio: cambia estos dos valores y vuelve a ejecutar (Shift+Enter)
MIS_CLUSTERS = 30
MI_TAMANO_MINIMO = 100

ventana = img[:, r0:r1, c0:c1]        # el acercamiento a la capital (21×21 km)
t0 = time.time()
mi_res = shepherd_wasm.doShepherdSegmentation(
    ventana, numClusters=MIS_CLUSTERS, minSegmentSize=MI_TAMANO_MINIMO,
    imgNullVal=nodata, fixedKMeansInit=True)
mi_seg = mi_res.segimg

bordes = (ndimage.maximum_filter(mi_seg, size=2)
          != ndimage.minimum_filter(mi_seg, size=2))
vis = rgb[r0:r1, c0:c1].copy(); vis[bordes] = [0, 1, 1]
plt.figure(figsize=(7, 7)); plt.imshow(vis)
plt.title(f"numClusters={MIS_CLUSTERS}, minSegmentSize={MI_TAMANO_MINIMO} → "
          f"{int(mi_seg.max()):,} objetos ({time.time()-t0:.1f} s)")
plt.axis("off"); plt.show()

---
## 🎓 Lo que acabas de lograr

1. Ejecutaste **Python científico completo dentro de tu navegador** — sin instalar nada, sin nube, sin permisos de administrador.
2. Procesaste **TODO el estado de Aguascalientes**: 12 millones de píxeles segmentados en cien mil objetos del territorio.
3. Entrenaste una **red neuronal apilada con árboles de decisión** usando la verdad-terreno oficial de INEGI — y produjiste el **mapa urbano estatal**, con su superficie estimada en km².
4. Entendiste la receta de los datos: **geomediana Sentinel-2** (10 m, 12 bandas, un año sin nubes) remuestreada a **calidad Landsat** (30 m, 6 bandas) para que el estado completo quepa en una pestaña.

**¿Y si quieres más resolución?** Ese es el siguiente nivel del curso: el **laboratorio portable (SatLab)** instala este mismo entorno en tu máquina, sin límite de 4 GB, con la geomediana Sentinel-2 completa (10 m, 12 bandas, 2.5 GB) — y ahí mismo puedes descargar imágenes frescas de tu zona desde Google Earth Engine. El navegador es tu aula; el portable, tu oficina.

---
*Implementación de segmentación validada bit a bit contra `pyshepseg` (ubarsc) — `pip install shepherd-wasm`. Datos: geomediana Sentinel-2 2020 (Copernicus / Digital Earth Africa-style), remuestreada a 30 m / 6 bandas; localidades del Marco Geoestadístico (INEGI). Plataforma: JupyterLite + Pyodide (WebAssembly). Código y cadena de verificación: [github.com/abxda/portable-satelital](https://github.com/abxda/portable-satelital).*